## Using MobileNetV2 for Face Mask Detection

> Importing libraries
---

In [ ]:
import pandas as pd
import os
import cv2
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

> Dataset Inspection
---

In [32]:
df = pd.read_csv("dataset/train/_classes.csv")

print(df.head())
print(df.columns.tolist())
print(df.shape)
print(df.sum(numeric_only=True))

                                            filename   with_mask  \
0  26-with-mask_jpg.rf.fb2a4fd14461ba4e7d981f51d9...           1   
1     66_jpg.rf.fb48333563aba0cd06afba343f625d98.jpg           0   
2  153-with-mask_jpg.rf.fb584ce29ce6d9a3123a04d80...           1   
3  29-with-mask_jpg.rf.fb9ecac8d3dc335a43b79a9438...           1   
4  augmented_image_315_jpg.rf.fba9c25c6675c5a039f...           1   

    without_mask  
0              0  
1              1  
2              0  
3              0  
4              0  
['filename', ' with_mask', ' without_mask']
(1279, 3)
with_mask       644
without_mask    635
dtype: int64


In [34]:
df.columns = df.columns.str.strip()

# Image directory
IMAGE_DIR = "dataset/train/"

# Check image availability
df["exists"] = df["filename"].apply(
    lambda x: os.path.exists(os.path.join(IMAGE_DIR, x))
)

print("Total images:", len(df))
print("Existing images:", df["exists"].sum())
print("Missing images:", (~df["exists"]).sum())

print("\nClass distribution:")
print(df[["with_mask", "without_mask"]].sum())

Total images: 1279
Existing images: 1279
Missing images: 0

Class distribution:
with_mask       644
without_mask    635
dtype: int64


> Train Test Split
---

In [36]:
IMAGE_DIR = "dataset/train"
IMG_SIZE = 160

df = pd.read_csv(os.path.join(IMAGE_DIR, "_classes.csv"))
df.columns = df.columns.str.strip()

X = []
y = []

for _, row in df.iterrows():
    img_path = os.path.join(IMAGE_DIR, row["filename"])

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    X.append(img)

    label = 1 if row["with_mask"] == 1 else 0
    y.append(label)

X = np.array(X, dtype=np.float32) / 255.0
y = np.array(y, dtype=np.int32)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)

Training: (1023, 160, 160, 3) (1023,)
Validation: (256, 160, 160, 3) (256,)


> Data Augmentation
---

In [37]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

# Data augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1)
])

# Base model
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(160, 160, 3)
)

base_model.trainable = False

# Model architecture
inputs = layers.Input(shape=(160, 160, 3))

x = data_augmentation(inputs)
x = layers.Rescaling(scale=2, offset=-1)(x)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 160, 160, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

> Model Training Optimization & Callbacks
---

In [38]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "best_mask_model.keras",
        monitor="val_loss",
        save_best_only=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    callbacks=callbacks
)

Epoch 1/20
 9/32 ━━━━━━━━━━━━━━━━━━━━ 14s 628ms/step - accuracy: 0.7014 - loss: 0.5809

KeyboardInterrupt: 

> Performance Matrix
---

In [39]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = (model.predict(X_val) > 0.5).astype(int)

print(classification_report(
    y_val,
    y_pred,
    target_names=["Without Mask", "With Mask"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))

8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 538ms/step
              precision    recall  f1-score   support

Without Mask       0.93      0.88      0.91       127
   With Mask       0.89      0.94      0.91       129

    accuracy                           0.91       256
   macro avg       0.91      0.91      0.91       256
weighted avg       0.91      0.91      0.91       256

Confusion Matrix:
[[112  15]
 [  8 121]]


> Saving Model
---

In [40]:
model.save("model/mask_detection_model.keras")

>Model Inference Speed Benchmarking
---

In [41]:
import time
import numpy as np

# Warm-up
for _ in range(10):
    model.predict(X_val[:1], verbose=0)

# Benchmark
start = time.perf_counter()

for _ in range(100):
    model.predict(X_val[:1], verbose=0)

end = time.perf_counter()

avg_time = (end - start) / 100

print(f"Average inference time: {avg_time * 1000:.2f} ms")
print(f"Estimated FPS: {1 / avg_time:.2f}")

Average inference time: 99.72 ms
Estimated FPS: 10.03


> Direct Model Inference Benchmarking
---

In [42]:
import time
import tensorflow as tf

sample = tf.convert_to_tensor(X_val[:1])

# Warm-up
for _ in range(20):
    model(sample, training=False)

# Benchmark
start = time.perf_counter()

for _ in range(100):
    model(sample, training=False)

# Ensure execution completes
_ = model(sample, training=False).numpy()

end = time.perf_counter()

avg_time = (end - start) / 100

print(f"Inference: {avg_time * 1000:.2f} ms")
print(f"FPS: {1 / avg_time:.2f}")

Inference: 200.32 ms
FPS: 4.99


> TFLite Model Conversion & Optimization
---

In [43]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open("model/mask_detection_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved!")

INFO:tensorflow:Assets written to: C:\Users\myper\AppData\Local\Temp\tmp6_l83373\assets


INFO:tensorflow:Assets written to: C:\Users\myper\AppData\Local\Temp\tmp6_l83373\assets


Saved artifact at 'C:\Users\myper\AppData\Local\Temp\tmp6_l83373'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 160, 160, 3), dtype=tf.float32, name='keras_tensor_320')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  1345173623568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345173624528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345173624336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345173623952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345173625104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345173622992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345173624720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345173624912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345173623184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1345173626064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1

>TFLite Inference Performance Benchmarking
---

In [ ]:
import numpy as np
import time
import tensorflow as tf

interpreter = tf.lite.Interpreter(
    model_path="model/mask_detection_model.tflite",
    num_threads=4
)

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

sample = X_val[:1].astype(np.float32)

interpreter.set_tensor(input_details[0]["index"], sample)

# Warm-up
for _ in range(20):
    interpreter.invoke()

start = time.perf_counter()

for _ in range(100):
    interpreter.set_tensor(input_details[0]["index"], sample)
    interpreter.invoke()

end = time.perf_counter()

avg_time = (end - start) / 100

print(f"Inference: {avg_time * 1000:.2f} ms")
print(f"FPS: {1 / avg_time:.2f}")